In [1]:
import os
import sys
from pathlib import Path

sys.path.append(str(Path(os.getcwd()).resolve()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from torchsummary import summary
from PIL import Image

import torch
import torch.nn.functional as F
import torch.optim as optim
import torch.nn as nn

import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.utils.data import DataLoader


In [2]:
from google.colab import drive

if not os.path.exists('/content/drive'):
    drive.mount('/content/drive', force_remount=True)
    print("Drive mounted successfully!")
else:
    print("Drive already mounted.")

Mounted at /content/drive
Drive mounted successfully!


### Clone git and load modules

In [3]:
!git clone https://github.com/gimoonnam/vgg16_practice.git

Cloning into 'vgg16_practice'...
remote: Enumerating objects: 20, done.
remote: Counting objects: 100% (20/20), done.
remote: Compressing objects: 100% (18/18), done.
remote: Total 20 (delta 8), reused 9 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (20/20), 12.85 KiB | 12.85 MiB/s, done.
Resolving deltas: 100% (8/8), done.


In [4]:
repo_path = '/content/vgg16_practice'
if repo_path not in sys.path:
  sys.path.insert(0, repo_path)

from load_data import CatDogDataLoadandSave
from vgg16_model import VGG16

### Load data and Save dataset as ubyte format


In [11]:
data_path = r'/content/drive/My Drive/Data for Colab Training'
train_data_path = os.path.join(data_path, "cat-and-dog", "training_set")
test_data_path  = os.path.join(data_path, "cat-and-dog", "test_set")


trainset = CatDogDataLoadandSave(train_data_path)
train_loader = DataLoader(trainset, batch_size=64, shuffle=True)


### Build VGG16 architecture

In [12]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = VGG16(3, 2).to(device)

device

device(type='cuda')

In [13]:
from torchsummary import summary
summary(model, (3, 224, 224))


----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 64, 224, 224]           1,792
              ReLU-2         [-1, 64, 224, 224]               0
            Conv2d-3         [-1, 64, 224, 224]          36,928
              ReLU-4         [-1, 64, 224, 224]               0
         MaxPool2d-5         [-1, 64, 112, 112]               0
            N_conv-6         [-1, 64, 112, 112]               0
            Conv2d-7        [-1, 128, 112, 112]          73,856
              ReLU-8        [-1, 128, 112, 112]               0
            Conv2d-9        [-1, 128, 112, 112]         147,584
             ReLU-10        [-1, 128, 112, 112]               0
        MaxPool2d-11          [-1, 128, 56, 56]               0
           N_conv-12          [-1, 128, 56, 56]               0
           Conv2d-13          [-1, 256, 56, 56]         295,168
             ReLU-14          [-1, 256,

In [18]:
learning_rate = 1e-4
num_epochs = 10

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

In [19]:
model.train()

for epoch in range(num_epochs):
    ProgressBar = tqdm(enumerate(train_loader), total=len(train_loader))

    for batch_idx, (inputs, labels) in ProgressBar:

        # Ensure labels are torch.long before moving to device for CrossEntropyLoss
        inputs, labels = inputs.to(device), labels.to(device)
        labels = labels.long()

        optimizer.zero_grad()
        outputs = model(inputs)

        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        #Update Progress bar
        ProgressBar.set_description(f'Epoch [{epoch+1}]')
        ProgressBar.set_postfix(loss=loss.item())

Epoch [10]: 100%|██████████| 126/126 [00:44<00:00,  2.81it/s, loss=0.0353]
